In [2]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# FUNCȚIA DE PREPROCESARE (Tight Crop + 512x512 + Ben Graham)
# ==========================================
def process_single_image(args):
    img_path, dest_path = args
    
    # Dacă poza a fost deja procesată (în caz de reluare script), dăm skip
    if os.path.exists(dest_path): 
        return True
    
    try:
        img = cv2.imread(img_path)
        if img is None: 
            return False
        
        # 1. Bounding Box & Tight Crop (Rapid)
        small_img = cv2.resize(img, (512, 512))
        gray_small = cv2.cvtColor(small_img, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray_small, 10, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            c = max(contours, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(c)
            scale_y, scale_x = img.shape[0] / 512, img.shape[1] / 512
            X, Y, W, H = int(x * scale_x), int(y * scale_y), int(w * scale_x), int(h * scale_y)
            
            center_x, center_y = X + W//2, Y + H//2
            radius = max(W, H) // 2
            start_x, end_x = max(0, center_x - radius), min(img.shape[1], center_x + radius)
            start_y, end_y = max(0, center_y - radius), min(img.shape[0], center_y + radius)
            
            img_cropped = img[start_y:end_y, start_x:end_x]
        else:
            img_cropped = img
            
        # 2. Resize la 512x512
        size = 512
        img_resized = cv2.resize(img_cropped, (size, size))
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
        
        # 3. Ben Graham
        blur = cv2.GaussianBlur(gray, (0, 0), size / 30.0)
        ben_graham = cv2.addWeighted(gray, 4, blur, -4, 128)
        
        # Salvăm direct la destinația stabilită
        cv2.imwrite(dest_path, cv2.cvtColor(ben_graham, cv2.COLOR_GRAY2BGR))
        return True
    
    except Exception as e:
        print(f"\n❌ Eroare la {os.path.basename(img_path)}: {e}")
        return False

# ==========================================
# EXECUTIA PRINCIPALA 
# ==========================================
if __name__ == '__main__':
    print("=== 🟠 SCRIPT SPLIT MULTI-CLASĂ (1, 2, 3, 4) - 70/15/15 ===")

    INPUT_CSV = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\originals\EyePACS\all_labels.csv'
    INPUT_DIR = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\originals\EyePACS\Images'
    OUT_DIR_MULTI = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_1_4'
    
    # Numele folderelor pentru clase
    clase_nume = {
        1: 'stadiul_1',
        2: 'stadiul_2',
        3: 'stadiul_3',
        4: 'stadiul_4'
    }

    # 1. Creare Structură Foldere
    for split in ['train', 'val', 'test']:
        for lvl, nume_folder in clase_nume.items():
            os.makedirs(os.path.join(OUT_DIR_MULTI, split, nume_folder), exist_ok=True)

    # 2. Încărcare și verificare
    print("\nCitim CSV-ul original...")
    df_tot = pd.read_csv(INPUT_CSV)
    df_tot['file_path'] = df_tot['image'].apply(lambda x: os.path.join(INPUT_DIR, str(x) + '.png')) 
    
    # FILTRĂM DOAR BOLNAVII (Clasele 1, 2, 3, 4)
    df = df_tot[df_tot['level'] > 0].copy()

    df['exists'] = df['file_path'].apply(os.path.exists)
    poze_gasite = df['exists'].sum()
    print(f"Am găsit {poze_gasite} imagini cu pacienți BOLNAVI valide pe disc.")
    if poze_gasite == 0:
        raise ValueError("Nu am găsit nicio imagine! Verifică extensia pozelor (.jpeg vs .jpg).")
    df = df[df['exists']].drop(columns=['exists']).reset_index(drop=True)

    # ==========================================
    # 3. ÎMPĂRȚIRE STRATIFICATĂ 70/15/15
    # ==========================================
    print("\nCalculăm matematica pentru Train (70%), Val (15%), Test (15%)...")
    
    # Tăiem 30% din total pentru Val + Test, lăsând 70% pentru Train
    df_train, df_temp = train_test_split(df, test_size=0.30, stratify=df['level'], random_state=42)
    
    # Din cei 30% rămași, tăiem pe jumătate (50%) -> adică 15% pentru Val și 15% pentru Test
    df_val, df_test = train_test_split(df_temp, test_size=0.50, stratify=df_temp['level'], random_state=42)

    df_train = df_train.copy(); df_train['split'] = 'train'
    df_val = df_val.copy();     df_val['split'] = 'val'
    df_test = df_test.copy();   df_test['split'] = 'test'

    df_final = pd.concat([df_train, df_val, df_test]).reset_index(drop=True)

    # ==========================================
    # 📊 AFIȘARE STATISTICI PENTRU AUGMENTARE
    # ==========================================
    print("\n" + "="*40)
    print("📊 DISTRIBUȚIA CLASELOR ÎN TRAIN SET (Atenție la dezechilibru!):")
    print("="*40)
    
    distrib = df_train['level'].value_counts().sort_index()
    max_class_count = distrib.max()
    
    for lvl, count in distrib.items():
        deficit = max_class_count - count
        mesaj_deficit = f" -> Necesită augmentare cu {deficit} imagini" if deficit > 0 else " -> (Clasa Majoritară)"
        print(f"  - Stadiul {lvl}: {count} imagini {mesaj_deficit}")
        
    print("="*40 + "\n")

    print(f"Total general Val: {len(df_val)} imagini")
    print(f"Total general Test: {len(df_test)} imagini")

    # ==========================================
    # 4. PROCESAREA EFECTIVĂ
    # ==========================================
    print("\nConstruim rutele de salvare...")
    tasks = []
    for _, row in df_final.iterrows():
        nivel = row['level']
        nume_folder = clase_nume[nivel]
        dest_path = os.path.join(OUT_DIR_MULTI, row['split'], nume_folder, os.path.basename(row['file_path']))
        tasks.append((row['file_path'], dest_path))

    print(f"🚀 Pornim procesarea pe {len(tasks)} fire de execuție...")
    cv2.setNumThreads(0)
    
    max_threads = min(12, (os.cpu_count() or 4) * 2) 
    
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        list(tqdm(executor.map(process_single_image, tasks), total=len(tasks)))
    
    print(f"\n✅ GATA! Datasetul MULTI-CLASĂ este generat și salvat în:\n{OUT_DIR_MULTI}")

=== 🟠 SCRIPT SPLIT MULTI-CLASĂ (1, 2, 3, 4) - 70/15/15 ===

Citim CSV-ul original...
Am găsit 23358 imagini cu pacienți BOLNAVI valide pe disc.

Calculăm matematica pentru Train (70%), Val (15%), Test (15%)...

📊 DISTRIBUȚIA CLASELOR ÎN TRAIN SET (Atenție la dezechilibru!):
  - Stadiul 1: 4343 imagini  -> Necesită augmentare cu 4863 imagini
  - Stadiul 2: 9206 imagini  -> (Clasa Majoritară)
  - Stadiul 3: 1461 imagini  -> Necesită augmentare cu 7745 imagini
  - Stadiul 4: 1340 imagini  -> Necesită augmentare cu 7866 imagini

Total general Val: 3504 imagini
Total general Test: 3504 imagini

Construim rutele de salvare...
🚀 Pornim procesarea pe 23358 fire de execuție...


  0%|          | 0/23358 [00:00<?, ?it/s]


❌ Eroare la 32253_right.png: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'


❌ Eroare la 43457_left.png: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'


✅ GATA! Datasetul MULTI-CLASĂ este generat și salvat în:
B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_1_4


In [3]:
import os
import cv2
import random
import numpy as np
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# FUNCȚIA DE AUGMENTARE PENTRU O SINGURĂ IMAGINE
# ==========================================
def process_augmentation(args):
    src_path, dest_path = args
    
    if os.path.exists(dest_path):
        return True
        
    try:
        img = cv2.imread(src_path)
        if img is None:
            return False
            
        # 1. Flip Orizontal Aleatoriu (50% șanse)
        if random.random() > 0.5:
            img = cv2.flip(img, 1)
            
        # 2. Flip Vertical Aleatoriu (50% șanse)
        if random.random() > 0.5:
            img = cv2.flip(img, 0)
            
        # 3. Rotație Aleatorie (între -15 și +15 grade)
        # Rotim în jurul centrului, iar zonele goale (marginile) rămân negre (borderValue=0)
        angle = random.uniform(-15.0, 15.0)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
        
        # Salvăm noua imagine augmentată
        cv2.imwrite(dest_path, img)
        return True
        
    except Exception as e:
        print(f"❌ Eroare la augmentarea pozei {src_path}: {e}")
        return False

# Calea către folderul de TRAIN pentru setul Multi-Clasă
TRAIN_DIR = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_1_4\train'

# Target-ul tău fix pe fiecare clasă
TARGET_COUNT = 10000

clase = ['stadiul_1', 'stadiul_2', 'stadiul_3', 'stadiul_4']
all_tasks = []

print("\nCalculăm deficitul pentru fiecare clasă din TRAIN...")

# 1. Analizăm fiecare folder și generăm sarcinile
for nume_clasa in clase:
    dir_clasa = os.path.join(TRAIN_DIR, nume_clasa)
    
    if not os.path.exists(dir_clasa):
        continue
        
    poze_existente = [f for f in os.listdir(dir_clasa) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    num_poze = len(poze_existente)
    
    deficit = TARGET_COUNT - num_poze
    
    if deficit > 0:
        print(f"📊 {nume_clasa}: {num_poze} poze. Generăm {deficit} imagini augmentate.")
        
        for i in range(deficit):
            poza_sursa = random.choice(poze_existente)
            src_path = os.path.join(dir_clasa, poza_sursa)
            
            nume_baza, ext = os.path.splitext(poza_sursa)
            dest_path = os.path.join(dir_clasa, f"{nume_baza}_aug_{i}{ext}")
            
            all_tasks.append((src_path, dest_path))
    else:
        print(f"✅ {nume_clasa}: are deja {num_poze} poze (Target atins).")

# 2. Execuția pe mai multe fire (ThreadPoolExecutor)
if all_tasks:
    print(f"\n🚀 Pornim augmentarea a {len(all_tasks)} imagini noi la comun...")
    
    cv2.setNumThreads(0)
    max_threads = min(12, (os.cpu_count() or 4) * 2)
    
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        list(tqdm(executor.map(process_augmentation, all_tasks), total=len(all_tasks)))
        
    print("\n🎉 GATA! Toate clasele din Train au acum fix 10.000 de imagini!")


Calculăm deficitul pentru fiecare clasă din TRAIN...
📊 stadiul_1: 4342 poze. Generăm 5658 imagini augmentate.
📊 stadiul_2: 9206 poze. Generăm 794 imagini augmentate.
📊 stadiul_3: 1461 poze. Generăm 8539 imagini augmentate.
📊 stadiul_4: 1340 poze. Generăm 8660 imagini augmentate.

🚀 Pornim augmentarea a 23651 imagini noi la comun...


  0%|          | 0/23651 [00:00<?, ?it/s]


🎉 GATA! Toate clasele din Train au acum fix 10.000 de imagini!
